# Chapter 1.1 — Wayback Machine: Dead Personal Web

**Dataset 2 of 3.** This notebook pulls snapshots from the Internet Archive Wayback Machine for three domains that, between them, constitute the *dead-personal-web stratum* of the internet:

- **geocities.com** — Yahoo's free homepage hosting platform, killed in 2009. The canonical archaeological layer of the early web.
- **xanga.com** — early-2000s teen blog platform, mostly defunct by 2013.
- **livejournal.com** — still alive but most Western users abandoned it around 2010; the archive holds an enormous mass of personal blogs whose authors are gone.

These three were chosen because (a) they are documented to be ruined platforms, (b) their content was overwhelmingly personal rather than commercial, and (c) their material survives only via the Wayback Machine — meaning every page we pull is, by definition, a digital remain.

**Target:** ≈ 85 snapshots per domain → ≥ 250 total.

**Method:**
1. Query the [Wayback CDX server](https://github.com/internetarchive/wayback/blob/master/wayback-cdx-server/README.md) for HTML snapshots of each domain in the personal-web era (1999–2010), filtered to `statuscode:200` and `mimetype:text/html`.
2. For each snapshot, fetch the raw archived HTML via `web.archive.org/web/<timestamp>id_/<url>` (the `id_` suffix returns the page without Wayback's UI chrome).
3. Extract `<title>`, body text (whitespace-cleaned), and the URL of the first inline `<img>` (used later as a low-fidelity thumbnail).
4. Drop pages with under ~40 characters of text (typical for redirect / parking pages).

**Output:** `data/raw/wayback/wayback_snapshots.csv`.

In [4]:
import sys, os, time, json, re
from pathlib import Path
from urllib.parse import urljoin

# Robust project-root finder: works whether JupyterLab launched the kernel
# from the project folder or from anywhere else on disk (e.g. Desktop).
def _find_project_root(marker="sa_utils.py"):
    p = Path.cwd().resolve()
    for c in [p] + list(p.parents):
        if (c / marker).exists(): return c
    cowork = Path.home() / "Library/Application Support/Claude/local-agent-mode-sessions"
    if cowork.exists():
        for hit in cowork.rglob(marker):
            return hit.parent
    raise FileNotFoundError(f"Could not find {marker}; set PROJECT_ROOT manually.")
PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from sa_utils import load_env, make_session, polite_sleep, DATA_RAW
import pandas as pd
from bs4 import BeautifulSoup

load_env()
OUT_DIR = DATA_RAW / "wayback"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUT_DIR}")

.env loaded. OPENAI_API_KEY present: True
Project root: /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs
Output dir:   /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/wayback


## 1. Query the CDX server for snapshot listings

`url=<domain>/*` matches any path under that domain. We restrict to 1999–2010 (the personal-web window), HTTP 200, and HTML mimetype. CDX returns a JSON-array-of-arrays where the first row is the header.

In [6]:
session = make_session()

CDX = "https://web.archive.org/cdx/search/cdx"

# Wayback's CDX server times out on broad wildcard queries like geocities.com/*
# so we scope each domain to multiple narrower sub-prefixes / year bands.
SCOPED_QUERIES = {
    "geocities.com": [
        ("geocities.com/Athens/*",        "1999", "2009"),
        ("geocities.com/SiliconValley/*", "1999", "2009"),
        ("geocities.com/SoHo/*",          "1999", "2009"),
        ("geocities.com/Hollywood/*",     "1999", "2009"),
        ("geocities.com/Paris/*",         "1999", "2009"),
        ("geocities.com/Tokyo/*",         "1999", "2009"),
        ("geocities.com/Heartland/*",     "1999", "2009"),
    ],
    "xanga.com": [
        ("xanga.com/*", "2002", "2004"),
        ("xanga.com/*", "2005", "2006"),
        ("xanga.com/*", "2007", "2008"),
    ],
    "livejournal.com": [
        ("livejournal.com/users/*",     "2002", "2005"),
        ("livejournal.com/community/*", "2002", "2005"),
        ("livejournal.com/users/*",     "2006", "2008"),
    ],
}

def cdx_search_scoped(prefix: str, yfrom: str, yto: str, timeout: int = 120):
    params = {
        "url":      prefix,
        "output":   "json",
        "from":     f"{yfrom}0101",
        "to":       f"{yto}1231",
        "filter":   ["statuscode:200", "mimetype:text/html"],
        "limit":    80,
        "collapse": "digest",
    }
    try:
        r = session.get(CDX, params=params, timeout=timeout)
    except Exception as e:
        return [], f"{type(e).__name__}: {str(e)[:80]}"
    if r.status_code != 200:
        return [], f"HTTP {r.status_code}"
    try:
        data = r.json()
    except Exception:
        return [], "non-JSON"
    if not data or len(data) < 2:
        return [], "0 rows"
    header, *body = data
    return [dict(zip(header, row)) for row in body], None

WANT_PER_DOMAIN = 85
listings = {}
for dom, queries in SCOPED_QUERIES.items():
    print(f"\n[{dom}] running {len(queries)} scoped queries (want {WANT_PER_DOMAIN}) ...")
    rows = []
    for prefix, yfrom, yto in queries:
        if len(rows) >= WANT_PER_DOMAIN:
            break
        chunk, err = cdx_search_scoped(prefix, yfrom, yto)
        if err:
            print(f"   {prefix:40s} {yfrom}-{yto}: skipped ({err})")
        else:
            print(f"   {prefix:40s} {yfrom}-{yto}: +{len(chunk)}  (running {len(rows) + len(chunk)})")
            rows.extend(chunk)
        polite_sleep(2.0)
    # dedupe by original URL, cap at target
    seen, dedup = set(), []
    for r in rows:
        u = r["original"]
        if u in seen: continue
        seen.add(u); dedup.append(r)
        if len(dedup) >= WANT_PER_DOMAIN: break
    listings[dom] = dedup
    print(f"   --> {dom} FINAL: {len(dedup)} unique snapshots")

TOTAL_LISTED = sum(len(v) for v in listings.values())
print(f"\nTOTAL listing rows: {TOTAL_LISTED}")


[geocities.com] running 7 scoped queries (want 85) ...
   geocities.com/Athens/*                   1999-2009: +80  (running 80)
   geocities.com/SiliconValley/*            1999-2009: +80  (running 160)
   --> geocities.com FINAL: 82 unique snapshots

[xanga.com] running 3 scoped queries (want 85) ...
   xanga.com/*                              2002-2004: +80  (running 80)
   xanga.com/*                              2005-2006: +80  (running 160)
   --> xanga.com FINAL: 3 unique snapshots

[livejournal.com] running 3 scoped queries (want 85) ...
   livejournal.com/users/*                  2002-2005: +80  (running 80)
   livejournal.com/community/*              2002-2005: +80  (running 160)
   --> livejournal.com FINAL: 63 unique snapshots

TOTAL listing rows: 148


In [7]:
# ---- Wayback safety top-up: only run if total < 200 ----
total = sum(len(v) for v in listings.values())
if total >= 200:
    print(f"Already have {total} unique URLs — no top-up needed.")
else:
    print(f"Only {total} unique URLs. Topping up from additional sources...")
    EXTRA_QUERIES = [
        ("geocities.com", "geocities.com/RodeoDrive/*",    "1999", "2009"),
        ("geocities.com", "geocities.com/Yosemite/*",      "1999", "2009"),
        ("geocities.com", "geocities.com/MotorCity/*",     "1999", "2009"),
        ("geocities.com", "geocities.com/Petsburgh/*",     "1999", "2009"),
        ("geocities.com", "geocities.com/BourbonStreet/*", "1999", "2009"),
        ("geocities.com", "geocities.com/TimesSquare/*",   "1999", "2009"),
        ("geocities.com", "geocities.com/Broadway/*",      "1999", "2009"),
        ("geocities.com", "geocities.com/CapeCanaveral/*", "1999", "2009"),
        ("angelfire.com", "angelfire.com/folk/*",          "2000", "2008"),
        ("angelfire.com", "angelfire.com/poetry/*",        "2000", "2008"),
        ("angelfire.com", "angelfire.com/ny/*",            "2000", "2008"),
        ("angelfire.com", "angelfire.com/journal/*",       "2000", "2008"),
    ]
    for dom, prefix, yfrom, yto in EXTRA_QUERIES:
        if sum(len(v) for v in listings.values()) >= 260:
            print("Reached 260 — stopping top-up.")
            break
        chunk, err = cdx_search_scoped(prefix, yfrom, yto)
        if err:
            print(f"  {prefix} {yfrom}-{yto}: skipped ({err})")
            continue
        listings.setdefault(dom, [])
        existing = {r["original"] for r in listings[dom]}
        added = 0
        for row in chunk:
            if row["original"] not in existing:
                listings[dom].append(row); existing.add(row["original"]); added += 1
        print(f"  {prefix} {yfrom}-{yto}: +{added} new (domain {dom} now {len(listings[dom])})")
        polite_sleep(2.0)
    total = sum(len(v) for v in listings.values())
    print(f"\nNew TOTAL listing rows: {total}")

Only 148 unique URLs. Topping up from additional sources...
  geocities.com/RodeoDrive/* 1999-2009: +30 new (domain geocities.com now 112)
  geocities.com/Yosemite/* 1999-2009: +33 new (domain geocities.com now 145)
  geocities.com/MotorCity/* 1999-2009: +27 new (domain geocities.com now 172)
  geocities.com/Petsburgh/* 1999-2009: +35 new (domain geocities.com now 207)
Reached 260 — stopping top-up.

New TOTAL listing rows: 273


## 2. Download each snapshot and extract text

The Wayback raw snapshot URL pattern is:
```
https://web.archive.org/web/<timestamp>id_/<original_url>
```
The trailing `id_` (note: 'identity' modifier — `id` + underscore) returns the archived page without Wayback's UI banner.

We rate-limit to ~1 request/sec to be courteous to the Internet Archive. Expect this cell to take ~5–8 minutes for 250 snapshots.

In [8]:
def fetch_snapshot(session, timestamp: str, original_url: str, timeout: int = 25) -> str | None:
    """Return raw HTML of an archived snapshot, or None on failure."""
    url = f"https://web.archive.org/web/{timestamp}id_/{original_url}"
    try:
        r = session.get(url, timeout=timeout)
    except Exception as e:
        return None
    if r.status_code != 200 or not r.text:
        return None
    return r.text

WS_RE = re.compile(r"\s+")

def extract_fields(html: str, snapshot_url: str) -> dict:
    soup = BeautifulSoup(html, "lxml") if html else None
    if soup is None:
        return {"title": "", "body_text": "", "first_img": None, "img_count": 0}
    # title
    title = (soup.title.string if soup.title and soup.title.string else "").strip()
    # remove scripts / styles / iframes etc. before reading text
    for tag in soup(["script", "style", "noscript", "iframe", "meta", "link"]):
        tag.decompose()
    body_text = WS_RE.sub(" ", soup.get_text(" ", strip=True)).strip()
    # truncate to ~3000 chars; we don't need novels and large fields blow up CSV
    body_text = body_text[:3000]
    # first inline image — try to keep an absolute Wayback URL
    imgs = [img.get("src") for img in soup.find_all("img") if img.get("src")]
    first_img = None
    if imgs:
        src = imgs[0]
        if src.startswith("//"):
            src = "https:" + src
        elif src.startswith("/"):
            src = urljoin("https://web.archive.org", src)
        elif not src.startswith("http"):
            src = urljoin(snapshot_url, src)
        first_img = src
    return {"title": title, "body_text": body_text, "first_img": first_img, "img_count": len(imgs)}

all_records = []
for dom, rows in listings.items():
    print(f"\n→ {dom} ({len(rows)} snapshots)")
    for i, row in enumerate(rows, 1):
        ts, orig = row["timestamp"], row["original"]
        snap_url = f"https://web.archive.org/web/{ts}/{orig}"
        html = fetch_snapshot(session, ts, orig)
        if not html:
            polite_sleep(0.8)
            continue
        fields = extract_fields(html, snap_url)
        all_records.append({
            "domain":        dom,
            "timestamp":     ts,
            "original_url":  orig,
            "snapshot_url":  snap_url,
            **fields,
            "html_length":   len(html),
        })
        if i % 10 == 0:
            print(f"   {i:3d}/{len(rows)}  collected so far: {len(all_records)}")
        polite_sleep(1.0)

print(f"\nTOTAL FETCHED: {len(all_records)}")


→ geocities.com (207 snapshots)
    10/207  collected so far: 10
    20/207  collected so far: 20
    30/207  collected so far: 30
    40/207  collected so far: 40
    50/207  collected so far: 50
    60/207  collected so far: 60
    70/207  collected so far: 70
    80/207  collected so far: 80
    90/207  collected so far: 90
   100/207  collected so far: 100
   110/207  collected so far: 110
   120/207  collected so far: 120
   130/207  collected so far: 130
   140/207  collected so far: 140
   150/207  collected so far: 150
   160/207  collected so far: 160
   170/207  collected so far: 170
   180/207  collected so far: 180
   190/207  collected so far: 190
   200/207  collected so far: 200

→ xanga.com (3 snapshots)

→ livejournal.com (63 snapshots)
    10/63  collected so far: 220
    20/63  collected so far: 230
    30/63  collected so far: 240
    40/63  collected so far: 250
    50/63  collected so far: 260
    60/63  collected so far: 270

TOTAL FETCHED: 273


In [9]:
df = pd.DataFrame(all_records)
before = len(df)
df = df[df["body_text"].str.len() >= 40].copy()
df["combined_text"] = (df["title"].fillna("") + "\n\n" + df["body_text"].fillna("")).str.strip()
df = df.reset_index(drop=True)
print(f"Filtered {before - len(df)} short/empty bodies. Remaining: {len(df)}")

csv_path  = OUT_DIR / "wayback_snapshots.csv"
json_path = OUT_DIR / "wayback_snapshots_raw.json"
df.to_csv(csv_path, index=False, encoding="utf-8")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

print(f"✓ Saved {len(df)} snapshots to {csv_path}")
print(f"✓ Saved raw dump to {json_path}")

Filtered 1 short/empty bodies. Remaining: 272
✓ Saved 272 snapshots to /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/wayback/wayback_snapshots.csv
✓ Saved raw dump to /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/wayback/wayback_snapshots_raw.json


## Quick stats

In [10]:
print("Per-domain counts (after filtering):")
print(df["domain"].value_counts().to_string())
print()
tok_lens = df["combined_text"].str.split().str.len()
print("Token length summary:")
print(tok_lens.describe().round(1).to_string())
print()
print("Snapshot timestamp range:")
ts = pd.to_datetime(df["timestamp"], format="%Y%m%d%H%M%S", utc=True, errors="coerce")
print(f"  earliest: {ts.min()}")
print(f"  latest:   {ts.max()}")
print()
print("How many have a first inline image:")
print(int(df["first_img"].notna().sum()), "/", len(df))

Per-domain counts (after filtering):
domain
geocities.com      206
livejournal.com     63
xanga.com            3

Token length summary:
count    272.0
mean     276.3
std      163.0
min       11.0
25%      142.8
50%      257.0
75%      433.2
max      610.0

Snapshot timestamp range:
  earliest: 1999-01-17 02:21:00+00:00
  latest:   2009-10-27 15:48:09+00:00

How many have a first inline image:
267 / 272


## Notes for the report

1. **CDX `collapse=digest` is conceptually important.** Wayback often holds dozens of near-identical captures of the same page. Collapsing on content hash means each row in our dataset is *one observed state of one page*, not redundant copies. This makes the corpus genuinely a set of remains rather than an over-sampled few.
2. **The `id_` modifier on the snapshot URL is essential.** Without it, the HTML we'd download would include the Wayback Machine's top banner / JS overlay — which would contaminate text vectorisation with strings like 'Wayback Machine' on every document.
3. **What we are deliberately *not* doing.** We are not de-noising the body text (e.g., removing nav menus, ads, footers). The thesis treats the *whole* page as the remain, including the boilerplate around the personal content. The wrongness of including boilerplate matches the project's whole methodology.
4. **Robustness caveat — same as Turin case.** The Wayback Machine's HTML structure and CDX server behaviour can change; re-running this scraper later may yield different results. Acknowledge in the writeup.